# WCB label transfer

Can labelled sentences from 24 other central banks stand in for scarce FOMC
labels? Three experiments, one model (roberta-large, winner config), all text
lowercased (WCB is lowercase-only -- casing must not masquerade as register).
Results land under corpus "twd-lc": comparisons are valid only inside this
lowercased room, against the lc control, never against the cased 0.707.

1. lc control: Shah train, lowercased -> prices the casing cost
2. transfer:   WCB only (zero FOMC labels) -> Shah test
3. augment:    Shah train + WCB           -> Shah test

In [ ]:
import os
import subprocess
import sys

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")


import polars as pl
import torch
from sklearn.metrics import f1_score

from config import RESULTS_DIR, SHAH_PLM
from data.loader_wcb_labelled import fetch_annotated
from data.loader_twd_labelled import load_splits
from models.plm_finetune import finetune, predict
from results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## Experiments 1-3

In [ ]:
# one seed first (chrono precedent) -- wcb epochs are ~10 min each, a 3-seed
# sweep is an overnight job. expand to SHAH_SEEDS once the shape is clear.
WCB_SEEDS = (78516,)

cfg = SHAH_PLM["roberta-large"]
wcb = pl.from_pandas(fetch_annotated()).select(
    pl.col("sentence").str.to_lowercase(),
    pl.col("label_int").alias("label"),
)
print(f"wcb: {len(wcb):,} rows")


def lc(df):
    return df.with_columns(pl.col("sentence").str.to_lowercase())


EXPS = ["roberta-large-lc", "wcb-only:roberta-large", "wcb-aug:roberta-large"]

for exp in EXPS:
    for seed in WCB_SEEDS:
        if already_done(OUT, force=FORCE, model=exp, corpus="twd-lc", seed=seed):
            print(f"{exp} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        train, test = lc(train).select("sentence", "label"), lc(test)
        val = None
        if exp == "wcb-only:roberta-large":
            train = wcb
        elif exp == "wcb-aug:roberta-large":
            # validate on Shah, not the mix: checkpoint selection must target FOMC
            val = train.sample(fraction=0.2, seed=seed)
            train = pl.concat(
                [train.filter(~pl.col("sentence").is_in(val["sentence"])), wcb]
            )
        print(f"{exp} seed {seed}: {len(train):,} training rows", flush=True)
        model, tok_, metrics = finetune(
            train,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test,
            val_df=val,
            device=DEVICE,
            verbose=True,
        )
        pred = predict(model, tok_, test, device=DEVICE)
        mf1 = f1_score(test["label"].to_list(), pred, average="macro")
        save_result(
            OUT,
            model=exp,
            corpus="twd-lc",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(mf1, 4),
        )
        print(f"{exp} seed {seed}: macro={mf1:.4f}")
        del model, tok_
        torch.cuda.empty_cache()